# Examine Relationship Between Combine Features And General Success During Hitch/In/Out Routes

**<span style="color: gold;">PROJECT</span>**

WR/DB Separation Analysis

**<span style="color: gold;">GOAL OF THIS NOTEBOOK</span>**

We have previously shown that deceleration-related Combine metrics have a significant relationship with an ability to create space during Hitch, In, and Out routes. We had to model that relationship using only 2023 Rookie data from the 2026 Big Data Bowl. 

Now, we want to examine if better deceleration performance is related an improvement in general WR success metrics. Since this doesn't require in-game tracking data for WRs and their defenders, we can use the entire rookie cohort from our 2027 Big Data Bowl data. Therefore, while the outcome we're looking at is more broad, our sample size has increased.

We will also examine our acceleration and cutting Combine metrics (not just deceleration Combine metrics).

Predictors: Combine Performance Metrics

Outcome: General WR Success Metrics (YAC, EPA, etc.)

**<span style="color: gold;">FINDINGS</span>**

There appear to be no significant relationships between our Combine metrics (or their composite scores) and completion percentage, completion percentage over expected, yards after catch, or EPA.


<br><br/>
#### *Initialize Environment*

In [ ]:
import sys
from pathlib import Path

# Identify Project Root
PROJECT_ROOT = next(
    parent
    for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "pyproject.toml").exists()
)
sys.path.insert(0, str(PROJECT_ROOT))

# Import Shared Project Paths
from utils.paths import (
    CLEAN_DATA_DIR,
    SEPARATION_DATA_DIR,
)

import polars as pl 
import pandas as pd 
import numpy as np
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "png"
from datetime import datetime

import statsmodels.api as sm

import nflreadpy as nfl

# Import schema_map For 2027 BDB Data
import sys
from pathlib import Path
# Identify Project Root

PROJECT_ROOT = next(

    parent

    for parent in [Path.cwd(), *Path.cwd().parents]

    if (parent / "pyproject.toml").exists()

)



# Import Shared Data Schemas

sys.path.insert(0, str(PROJECT_ROOT))

from utils.schema import schema_map

<br><br/>
## Load Data
***

#### Set Parameters For Analysis

In [ ]:
# =======================================
# Import Parameters That Define What 
# Data We'll Examine In This Project
# =======================================
param_path = PROJECT_ROOT / "05_WR_Separation_Analysis" / "Data_Pull"
sys.path.append(str(param_path))
import Data_Parameters as params

In [ ]:
# =======================================
# Specify Routes of Interest
# =======================================

ROUTES_OF_INTEREST = ["HITCH", "IN", "OUT"]

<br><br/>
#### WR Player-Plays From Rookie Cohort

In [ ]:
# =============================
# Get Rookie WRs
# =============================

players_of_interest = (
    pl.read_csv(
        str(CLEAN_DATA_DIR / "players.csv"), 
        schema_overrides = schema_map["players"]
    )
    # ------------------------------
    # Only Keep Position of Interest
    # ------------------------------
    .filter(pl.col("nfl_position") == "WR")
    # ------------------------------
    # Get List of nfl_ids
    # ------------------------------
    .select("nfl_id")
    .unique()
    .to_series()
    .to_list()
)

# ==============================
# Get Player-Play Data
# ==============================

player_play_df = (
    pl.read_csv(
        str(CLEAN_DATA_DIR / "player_play.csv"), 
        schema_overrides = schema_map["player_play"]
    )
    # ----------------------------
    # Only Keep Players of Interest
    # ----------------------------
    .filter(pl.col("nfl_id").is_in(players_of_interest))
    # ----------------------------
    # Only Keep Neeeded Variables
    # ----------------------------
    .select(
        "game_id", 
        "play_id", 
        "nfl_id", 
        "play_direction", 
        "yardline_number", 
        "target",
        "route_ran", 
        "team_coverage_man_zone",
        "separation_at_pass_forward",
        "pass_result"
    )
)

player_play_df

<br><br/>
#### General WR Performance Data

In [ ]:
# ====================================
# Get General WR Performance Data
# ====================================

wr_performance_df = (
    nfl.load_pbp(seasons=[2023, 2024, 2025])
    # -------------------------------------
    # Clean old_game_id and play_id
    # -------------------------------------
    .with_columns(
        game_id = pl.col("old_game_id").cast(pl.Int64), 
        play_id = pl.col("play_id").cast(pl.Int64)
    )
    # -------------------------------------
    # Only Keep Needed Variables
    # -------------------------------------
    .select(
        "game_id", 
        "play_id", 
        "cp", 
        "cpoe", 
        "yards_after_catch", 
        "epa"
    )
)

print("General WR Performance Data:")
print(f"    - One Row Per Play (Will Have To Filter To Our Targeted Player-Plays Later)")
wr_performance_df

<br><br/>
#### Combine Features Data

In [ ]:
# =======================================
# Combine Features Data
# =======================================

combine_features_df = pl.read_csv(str(SEPARATION_DATA_DIR / "03_WR_Combine_Features_Data.csv"))

print("Combine Features Data:")
print(f"    - Dimensions: {combine_features_df.shape}")
print(f"    - Number of Players: {combine_features_df.select("nfl_id").n_unique()}")
combine_features_df

<br><br/>
## Clean Data
***

#### Join Route Data and General Performance Data

In [ ]:
# ====================================
# Combine Route and Performance Datasets
# ====================================

route_and_performance_df = (
    player_play_df
    # ------------------------------
    # Ignore Players Who Weren't Targeted
    # ------------------------------
    .filter(pl.col("target"))
    # ------------------------------
    # Join Performance Data
    # ------------------------------
    .join(
        wr_performance_df, 
        on = ["game_id", "play_id"], 
        how = "inner"
    )
)

route_and_performance_df

<br><br/>
#### Summarize Player Performance On Routes of Interest

In [ ]:
route_and_performance_df = (
    route_and_performance_df
    # -----------------------------
    # Only Consider Routes of Interest
    # -----------------------------
    .filter(pl.col("route_ran").is_in(ROUTES_OF_INTEREST))
    # -----------------------------
    # Calculate Avg CPOE For Each Player Running Those Routes Of Interest
    # -----------------------------
    .group_by("nfl_id")
    .agg(
        avg_cp = pl.mean("cp"),
        avg_cpoe = pl.mean("cpoe"),
        avg_yac = pl.mean("yards_after_catch"), 
        avg_epa = pl.mean("epa")
    )
)

route_and_performance_df

<br><br/>
#### Join Combine Features Data

In [ ]:
analysis_df = (
    route_and_performance_df
    .join(
        combine_features_df, 
        on = "nfl_id", 
        how = "inner"
    )
)

analysis_df

<br><br/>
## Examine Associations
***

In [ ]:
predictor = "master_agility_score"
outcome = "avg_yac" 

fig = px.scatter(
    data_frame = analysis_df, 
    x = predictor, 
    y = outcome, 
    trendline = "ols"
)

fig.show()

In [ ]:
regression_df = analysis_df

regression_df = regression_df.filter(
    (pl.col(predictor).is_not_null())
    & 
    (pl.col(outcome).is_not_null())

)

pd_reg = regression_df.to_pandas()

X = pd_reg[predictor]
y = pd_reg[outcome]

X = sm.add_constant(X)

model = sm.OLS(y, X).fit()
print(model.summary())